# Notebook 1: Getting started with QSARmil (the easy way)

This notebook is for anyone who just wants to build a model and get predictions, without needing to know how the
model works under the hood. You give QSARmil a list of molecules (as SMILES strings) and their measured property
(e.g. how active they are against a target), and QSARmil takes care of everything else: generating conformers,
computing descriptors, training several models, and picking the best combination.

If you later want to understand or customize each of these steps yourself, see
`02_Professional_Pipeline_Customization.ipynb`.

In [ ]:
import pandas as pd
from sklearn.metrics import r2_score

from qsarmil import MultiConformerRegressor, MultiConformerClassifier

### 1. Load your data

QSARmil expects two things: a list of molecules written as SMILES strings, and a list of the property values you
want to predict (one number per molecule).

As an example, we use a small, publicly available dataset of measured binding activities, from:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine
> learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

In [ ]:
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]].reset_index(drop=True)
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]].reset_index(drop=True)
df_train.shape, df_test.shape

Training a full model on the whole dataset can take a while, since QSARmil tries many different model
combinations. If you just want to try things out quickly (for example, to check that everything runs on your
machine), keep the cell below **uncommented** to work with a small random sample instead. Comment it out once
you're ready to run the real thing.

In [ ]:
# uncomment to work with a small sample instead of the full dataset (much faster, good for a first try)
df_train = df_train.sample(n=15, random_state=42).reset_index(drop=True)
df_test = df_test.sample(n=5, random_state=42).reset_index(drop=True)
df_train.shape, df_test.shape

### 2. Build a multi-conformer model

Behind the scenes, QSARmil does the following for you:

1. **Generate conformers:** each molecule can fold into several different 3D shapes (conformers), so QSARmil
   generates a handful of them for each molecule.
2. **Compute descriptors:** each conformer is turned into a list of numbers (a descriptor) that a model can learn
   from. QSARmil tries several different kinds of descriptors.
3. **Train models:** several different modeling methods are trained, one for each descriptor type.
4. **Pick the best combination:** a search finds the best-performing combination ("consensus") of the models
   trained in step 3.

Two classes are available, depending on what you're predicting:

- **`MultiConformerRegressor`** - for a continuous property (e.g. binding affinity, solubility, an IC50 value).
- **`MultiConformerClassifier`** - for a yes/no property (e.g. active vs. inactive).

You always have to pick one explicitly - QSARmil never tries to guess which one you meant from your data, since a
column of 0s and 1s could just as easily be a real (if narrow-ranged) measurement rather than a class label.

Both classes accept the same settings when you create them:

- `num_conf` - how many conformers to generate per molecule. More conformers can capture more information, but
  takes longer. Default: `10`.
- `hopt` - whether QSARmil should automatically search for better model settings for every model it trains.
  Improves accuracy but takes noticeably longer. Default: `False`.
- `num_cpu` - how many CPU threads to use while generating conformers. Default: all available CPUs.
- `output_folder` - where QSARmil writes its files (`train.csv`/`val.csv`/`test.csv`). Default: a timestamped
  folder created automatically, e.g. `qsarmil_27_08_2026_23_09_47`.
- `verbose` - whether to print progress while training (which step it's on, how far along it is). Default: `True`.
- `random_seed` - a fixed random number so that re-running the same code gives the same result. Default: `42`.
- `accelerator` - whether to train on `"cpu"` or `"gpu"`. This is never guessed automatically - you choose.
  Default: `"cpu"`.

There's only one method you need, and it does everything in one call - training, model selection, and prediction:

- **`model.train_predict(smiles_train, y_train, smiles_test)`** - trains the model on your data and returns
  predictions for `smiles_test`. There's no separate save/load step and no separate `predict()` call, so the test
  SMILES you want predictions for have to be passed in up front, alongside the training data.

Let's build one below, spelling out every parameter explicitly so you can see what each one does.

In [ ]:
smiles_train, y_train = df_train["smiles"].to_list(), df_train["y"].to_list()
smiles_test = df_test["smiles"].to_list()

model = MultiConformerRegressor(
    num_conf=10,          # generate up to 10 conformers per molecule
    hopt=False,            # skip automatic hyperparameter search, for speed
    num_cpu=4,              # use 4 CPU threads for conformer generation
    output_folder=None,    # let QSARmil create a timestamped folder automatically
    verbose=True,           # print progress while training
    random_seed=42,         # fixed random seed, for reproducible results
    accelerator="cpu",      # train on CPU ("gpu" is also available if you have one)
)
y_pred = model.train_predict(smiles_train, y_train, smiles_test)

### 3. Check the predictions

`train_predict` already returned predictions for `smiles_test` above, made using the model it just trained - there's
no separate save/load step, so training and prediction happen together in one call. All we need here is the true
property values for `smiles_test`, so we can check how good the predictions are.

In [ ]:
y_test = df_test["y"].to_list()

In [ ]:
r2_score(y_test, y_pred)

### What's next?

- For a binary yes/no property, use `MultiConformerClassifier` instead - everything else about this notebook stays
  the same.
- If you'd like to understand (or customize) each of the steps QSARmil does for you automatically - the conformer
  generation, the descriptors, the modeling methods - see `02_Professional_Pipeline_Customization.ipynb`.
- If you're interested in figuring out *which* conformer a model thinks matters most for its prediction, see
  `03_Key_Instance_Detection.ipynb`.